In [37]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report

In [38]:
df = pd.read_csv('movies.csv')
df.head()

,Id,Title,Genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [39]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Id      9742 non-null   int64 
 1   Title   9742 non-null   object
 2   Genres  9742 non-null   object
dtypes: int64(1), object(2)
memory usage: 228.5+ KB


In [40]:
df.duplicated().sum()

0

In [41]:
df.shape

(9742, 3)

In [42]:
df['Year'] = df['Title'].str.extract(r'\((\d{4})\)').astype(float)
df.head()

,Id,Title,Genres,Year
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995.0
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995.0
2,3,Grumpier Old Men (1995),Comedy|Romance,1995.0
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995.0
4,5,Father of the Bride Part II (1995),Comedy,1995.0


In [43]:
df['NewTitle'] = df["Title"].str.replace(r'\(\d{4}\)', '', regex=True).str.strip()

df.head()

,Id,Title,Genres,Year,NewTitle
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995.0,Toy Story
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995.0,Jumanji
2,3,Grumpier Old Men (1995),Comedy|Romance,1995.0,Grumpier Old Men
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995.0,Waiting to Exhale
4,5,Father of the Bride Part II (1995),Comedy,1995.0,Father of the Bride Part II


In [44]:
df["GenreList"] = df["Genres"].str.split("|")
df.head()

,Id,Title,Genres,Year,NewTitle,GenreList
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995.0,Toy Story,"[Adventure, Animation, Children, Comedy, Fantasy]"
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995.0,Jumanji,"[Adventure, Children, Fantasy]"
2,3,Grumpier Old Men (1995),Comedy|Romance,1995.0,Grumpier Old Men,"[Comedy, Romance]"
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995.0,Waiting to Exhale,"[Comedy, Drama, Romance]"
4,5,Father of the Bride Part II (1995),Comedy,1995.0,Father of the Bride Part II,[Comedy]


In [50]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(df['GenreList'])

genre_df = pd.DataFrame(genre_matrix, columns=mlb.classes_)
df = pd.concat([df, genre_df], axis=1)

tfidf = TfidfVectorizer(stop_words='english')
X = tfidf.fit_transform(df['NewTitle'])
y = mlb.fit_transform(df['GenreList'])

In [51]:
df.head()

,Id,Title,Genres,Year,NewTitle,GenreList,(no genres listed),Action,Adventure,Animation,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995.0,Toy Story,"[Adventure, Animation, Children, Comedy, Fantasy]",0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995.0,Jumanji,"[Adventure, Children, Fantasy]",0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
2,3,Grumpier Old Men (1995),Comedy|Romance,1995.0,Grumpier Old Men,"[Comedy, Romance]",0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995.0,Waiting to Exhale,"[Comedy, Drama, Romance]",0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
4,5,Father of the Bride Part II (1995),Comedy,1995.0,Father of the Bride Part II,[Comedy],0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [52]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [53]:
model = OneVsRestClassifier(LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)

OneVsRestClassifier(estimator=LogisticRegression(max_iter=1000))

In [54]:
y_pred = model.predict(X_test)

In [55]:
print(classification_report(y_test, y_pred, target_names=mlb.classes_))

                    precision    recall  f1-score   support

(no genres listed)       0.00      0.00      0.00         7
            Action       0.81      0.07      0.13       367
         Adventure       0.60      0.02      0.05       253
         Animation       0.75      0.02      0.04       136
          Children       1.00      0.01      0.01       135
            Comedy       0.65      0.25      0.36       727
             Crime       0.67      0.01      0.02       245
       Documentary       0.00      0.00      0.00        79
             Drama       0.58      0.33      0.42       865
           Fantasy       0.00      0.00      0.00       150
         Film-Noir       0.00      0.00      0.00        13
            Horror       0.86      0.03      0.05       215
              IMAX       0.00      0.00      0.00        37
           Musical       0.00      0.00      0.00        75
           Mystery       0.00      0.00      0.00       101
           Romance       0.50      0.02

c:\users\aik\appdata\local\programs\python\python37\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\users\aik\appdata\local\programs\python\python37\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [56]:
def predict_genres(title):
    title_clean = tfidf.transform([title])
    pred = model.predict(title_clean)
    return mlb.inverse_transform(pred)

print(predict_genres("Toy Story"))

[('Comedy',)]
